In [1]:
"""
01 - TRAIN PCA MODEL
Fits a 13-band PCA model using ALL training patches only.
Run in Kaggle after attaching the DIPCOMA train/validation dataset.
"""

import os
import json
import numpy as np
import pandas as pd

BASE = "/kaggle/input/datasets/krrishshandilya/dipcoma-trainval"
DATASET = os.path.join(BASE, "dataset")
RESULTS = "/kaggle/working/pca_results"
os.makedirs(RESULTS, exist_ok=True)

def slugify(value):
    return str(value).strip().lower().replace(" ", "_").replace("-", "_")

# Read normalization limits from the dataset protocol.
with open(os.path.join(BASE, "protocol.json"), "r") as f:
    protocol = json.load(f)

LO = np.asarray(protocol["lo"], dtype=np.float64)
HI = np.asarray(protocol["hi"], dtype=np.float64)
assert len(LO) == 13 and len(HI) == 13
assert np.all(HI > LO)

patch_log = pd.read_csv(os.path.join(BASE, "patch_log.csv"))
train_rows = patch_log[patch_log["split"].str.lower() == "train"].copy()

def get_patch_path(row):
    region_folder = slugify(row["region"])
    return os.path.join(DATASET, "train", region_folder, row["patch"])

train_paths = [get_patch_path(row) for _, row in train_rows.iterrows()]
missing = [p for p in train_paths if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError(
        f"{len(missing)} training patches are missing. First missing path: {missing[0]}"
    )

print("Training patches:", len(train_paths))
print("Normalization: x = (clip(DN, LO, HI) - LO) / (HI - LO)")

# Pass 1: calculate the global mean across every pixel in every training patch.
pixel_sum = np.zeros(13, dtype=np.float64)
total_pixels = 0

for i, path in enumerate(train_paths, start=1):
    raw = np.load(path).astype(np.float32)
    normalized = (np.clip(raw, LO[:, None, None], HI[:, None, None])
                  - LO[:, None, None]) / (HI[:, None, None] - LO[:, None, None])
    X = normalized.reshape(13, -1).T.astype(np.float64)
    pixel_sum += X.sum(axis=0)
    total_pixels += X.shape[0]

    if i % 100 == 0 or i == len(train_paths):
        print(f"Mean pass: {i}/{len(train_paths)}")

mean = pixel_sum / total_pixels

# Pass 2: calculate covariance around the global mean.
covariance_sum = np.zeros((13, 13), dtype=np.float64)

for i, path in enumerate(train_paths, start=1):
    raw = np.load(path).astype(np.float32)
    normalized = (np.clip(raw, LO[:, None, None], HI[:, None, None])
                  - LO[:, None, None]) / (HI[:, None, None] - LO[:, None, None])
    X = normalized.reshape(13, -1).T.astype(np.float64)
    centered = X - mean
    covariance_sum += centered.T @ centered

    if i % 100 == 0 or i == len(train_paths):
        print(f"Covariance pass: {i}/{len(train_paths)}")


covariance = covariance_sum / total_pixels

# Eigenvectors are sorted by descending eigenvalue.
eigenvalues, eigenvectors = np.linalg.eigh(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
components = eigenvectors[:, order].T

explained_variance = eigenvalues / eigenvalues.sum()
cumulative_variance = np.cumsum(explained_variance)

model_path = os.path.join(RESULTS, "pca_model.npz")
np.savez(
    model_path,
    mean=mean.astype(np.float32),
    components=components.astype(np.float32),
    eigenvalues=eigenvalues.astype(np.float64),
    explained_variance=explained_variance.astype(np.float64),
    cumulative_variance=cumulative_variance.astype(np.float64),
    LO=LO.astype(np.float32),
    HI=HI.astype(np.float32),
)

print("\nPCA MODEL SAVED:", model_path)
print("Pixels used:", total_pixels)
print("\nExplained variance:")
for i, (ev, cv) in enumerate(zip(explained_variance, cumulative_variance), start=1):
    print(f"PC{i:02d}: {ev*100:.4f}% | cumulative: {cv*100:.4f}%")


Training patches: 957
Normalization: x = (clip(DN, LO, HI) - LO) / (HI - LO)
Mean pass: 100/957
Mean pass: 200/957
Mean pass: 300/957
Mean pass: 400/957
Mean pass: 500/957
Mean pass: 600/957
Mean pass: 700/957
Mean pass: 800/957
Mean pass: 900/957
Mean pass: 957/957
Covariance pass: 100/957
Covariance pass: 200/957
Covariance pass: 300/957
Covariance pass: 400/957
Covariance pass: 500/957
Covariance pass: 600/957
Covariance pass: 700/957
Covariance pass: 800/957
Covariance pass: 900/957
Covariance pass: 957/957

PCA MODEL SAVED: /kaggle/working/pca_results/pca_model.npz
Pixels used: 250871808

Explained variance:
PC01: 69.6792% | cumulative: 69.6792%
PC02: 16.6651% | cumulative: 86.3443%
PC03: 8.5803% | cumulative: 94.9247%
PC04: 3.5482% | cumulative: 98.4728%
PC05: 0.9028% | cumulative: 99.3756%
PC06: 0.3153% | cumulative: 99.6909%
PC07: 0.1101% | cumulative: 99.8010%
PC08: 0.1008% | cumulative: 99.9018%
PC09: 0.0386% | cumulative: 99.9404%
PC10: 0.0310% | cumulative: 99.9714%
PC11: 0